# PyCaret — Capabilities Landscape
### Part 5

[PyCaret](https://pycaret.org/) is a low-code AutoML library that wraps scikit-learn, XGBoost,
LightGBM and more behind a handful of functions. This notebook tours its breadth across
**classification, regression, clustering, and anomaly detection**.

**Covered:**
1. `setup()` — automated preprocessing pipeline
2. **Classification**: `compare_models`, `tune_model`, `plot_model`, `interpret_model`
3. **Regression**: end-to-end
4. **Clustering**: unsupervised module
5. **Anomaly detection**
6. `finalize_model` / `predict_model` / `save_model`

> Runtime: CPU is enough. `Runtime → Run all`.

In [ ]:
!pip install -q pycaret
print("Installed. Restart runtime if prompted, then Run all.")

In [ ]:
import pycaret
print("PyCaret version:", pycaret.__version__)

## 1. Classification

PyCaret ships sample datasets. We'll use `diabetes` (binary classification).
`setup()` builds the full preprocessing pipeline (imputation, encoding, scaling, split).

In [ ]:
from pycaret.datasets import get_data
from pycaret.classification import (setup, compare_models, tune_model, plot_model,
                                    interpret_model, finalize_model, predict_model,
                                    save_model, evaluate_model)

clf_data = get_data("diabetes")

In [ ]:
s = setup(data=clf_data, target="Class variable", session_id=42, verbose=False)
print("Setup complete — preprocessing pipeline ready.")

### 1a. Compare models — the leaderboard

`compare_models()` trains ~15 algorithms with cross-validation and ranks them.

In [ ]:
best = compare_models()

### 1b. Tune the best model (hyperparameter search)

In [ ]:
tuned = tune_model(best)

### 1c. Diagnostic plots

In [ ]:
plot_model(tuned, plot="auc")
plot_model(tuned, plot="confusion_matrix")
plot_model(tuned, plot="feature")

### 1d. Finalize & predict

In [ ]:
final_clf = finalize_model(tuned)
preds = predict_model(final_clf, data=clf_data.drop(columns=["Class variable"]).head())
preds

## 2. Regression

Same low-code pattern, numeric target. Using the `insurance` dataset (predict charges).

In [ ]:
from pycaret.regression import (setup as rsetup, compare_models as rcompare,
                                 plot_model as rplot, finalize_model as rfinal,
                                 predict_model as rpredict)

reg_data = get_data("insurance")
rs = rsetup(data=reg_data, target="charges", session_id=42, verbose=False)
best_reg = rcompare()

In [ ]:
rplot(best_reg, plot="residuals")
rplot(best_reg, plot="error")

## 3. Clustering (unsupervised)

The clustering module assigns cluster labels and plots them. We cluster the `jewellery` dataset.

In [ ]:
from pycaret.clustering import (setup as csetup, create_model as ccreate,
                                 assign_model, plot_model as cplot)

clu_data = get_data("jewellery")
cs = csetup(data=clu_data, session_id=42, verbose=False)
kmeans = ccreate("kmeans", num_clusters=4)
clustered = assign_model(kmeans)
clustered.head()

In [ ]:
cplot(kmeans, plot="cluster")
cplot(kmeans, plot="elbow")

## 4. Anomaly detection

Flag outliers using Isolation Forest via PyCaret's anomaly module.

In [ ]:
from pycaret.anomaly import (setup as asetup, create_model as acreate,
                              assign_model as aassign, plot_model as aplot)

anom = asetup(data=clu_data, session_id=42, verbose=False)
iforest = acreate("iforest")
anomalies = aassign(iforest)
print("Anomalies flagged:", int(anomalies["Anomaly"].sum()), "of", len(anomalies))
anomalies.head()

## 5. Save a model for deployment

In [ ]:
save_model(final_clf, "diabetes_pipeline")
print("Saved diabetes_pipeline.pkl (full preprocessing + model)")

## Summary

PyCaret compresses the ML workflow into a few calls across **every** task type:
`setup → compare_models → tune_model → plot_model → finalize_model → predict_model`.
It handled preprocessing automatically and gave a cross-validated leaderboard for classification,
regression, clustering, and anomaly detection — all with the same consistent API.